In [1]:
import os, json, random, collections
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

random.seed(42)
BDD = '/kaggle/input/datasets/solesensei/solesensei_bdd100k'
LAB = BDD + '/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_%s.json'
IMG = BDD + '/bdd100k/bdd100k/images/100k/%s'
OUT = '/kaggle/working/data'
CAP, MIN_AREA, MIN_CLASS = 1500, 3000, 200

def save(src, dst, size):
    try:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        Image.open(src).convert('RGB').resize(size).save(dst, quality=92)
        return True
    except Exception as e:
        print('skip', src, e)
        return False

# index every image under train/ and val/ (train is split into subfolders)
idx = {}
for sp in ['train', 'val']:
    for r, _, fs in os.walk(IMG % sp):
        for f in fs:
            if f.endswith('.jpg'):
                idx[(sp, f)] = os.path.join(r, f)
print('images on disk', len(idx))

W = {'clear': 'clear', 'partly cloudy': 'clear', 'overcast': 'overcast', 'rainy': 'rainy', 'snowy': 'snow_fog', 'foggy': 'snow_fog'}
T = {'daytime': 'day', 'night': 'night', 'dawn/dusk': 'twilight'}
pool, missing = [], 0
for sp in ['train', 'val']:
    for x in json.load(open(LAB % sp)):
        a = x['attributes']
        if a['weather'] not in W or a['timeofday'] not in T:
            continue
        src = idx.get((sp, x['name']))
        if src is None:
            missing += 1
            continue
        ps = [(b['box2d']['x2'] - b['box2d']['x1']) * (b['box2d']['y2'] - b['box2d']['y1'])
              for b in x['labels'] if b['category'] in ('person', 'rider') and 'box2d' in b]
        if ps and max(ps) < MIN_AREA:   # only tiny people: ambiguous, skip
            continue
        pool.append((src, W[a['weather']] + '_' + T[a['timeofday']], int(bool(ps))))
print('labeled but missing on disk', missing)
by = collections.defaultdict(list)
for p in pool:
    by[p[1]].append(p)
print({k: len(v) for k, v in sorted(by.items())})
classes = sorted(k for k, v in by.items() if len(v) >= MIN_CLASS)
sel = []
for k in classes:
    v = by[k]; random.shuffle(v); n = min(len(v), CAP)
    for i, p in enumerate(v[:n]):
        sp = 'train' if i < 0.83 * n else 'val' if i < 0.915 * n else 'test'
        sel.append((p[0], f'bdd/{sp}/{os.path.basename(p[0])}', k, classes.index(k), p[2], sp))
bd = pd.DataFrame(sel, columns=['src', 'path', 'cond', 'cond_id', 'person', 'split'])
with ThreadPoolExecutor(16) as ex:
    ok = list(ex.map(lambda r: save(r[0], f'{OUT}/{r[1]}', (384, 216)), sel))
bd = bd[ok]
bd.drop(columns='src').to_csv('/kaggle/working/bdd_manifest.csv', index=False)
json.dump({'cond': classes}, open('/kaggle/working/bdd_class_names.json', 'w'))
print('BDD', len(bd)); print(bd.split.value_counts())
print(bd.groupby('cond').size()); print('person rate', bd.person.mean())

images on disk 80000
labeled but missing on disk 0
{'clear_day': 14829, 'clear_night': 23089, 'clear_twilight': 2467, 'overcast_day': 6495, 'overcast_night': 78, 'overcast_twilight': 1043, 'rainy_day': 2354, 'rainy_night': 2196, 'rainy_twilight': 322, 'snow_fog_day': 2558, 'snow_fog_night': 2240, 'snow_fog_twilight': 422}
BDD 13787
split
train    11445
val       1176
test      1166
Name: count, dtype: int64
cond
clear_day            1500
clear_night          1500
clear_twilight       1500
overcast_day         1500
overcast_twilight    1043
rainy_day            1500
rainy_night          1500
rainy_twilight        322
snow_fog_day         1500
snow_fog_night       1500
snow_fog_twilight     422
dtype: int64
person rate 0.18481177921230144
